### Scikit-Learn Pipelines:
* Scikit-Learn Pipelines allows us to chain the the operation and techniques we need to apply in the dataset.
* Think of it like making function calls automatically, but on steroids.
* Kind of like making a assembly line in the factory.

* Pipeline allows you to sequentially apply a list of transformers to preprocess the data and, if desired, conclude the sequence with a final predictor for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

In [ ]:
titanic_df = pd.read_csv(filepath_or_buffer = '~/ML/Datasets/titanic.csv')

In [ ]:
titanic_df.drop(columns = ['Name', 'PassengerId', 'Ticket', 'Cabin'], inplace = True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(titanic_df.drop(columns = ['Survived']), titanic_df['Survived'], test_size = 0.2, random_state = 42)

In [ ]:
titanic_df.sample(5)

In [ ]:
titanic_df['Sex'].unique()

In [ ]:
X_train.columns

#### Steps that the pipline will have:
1. Deal with missing values by imputation
2. Encode the categorical data 
3. Deal with the outliers (Optional here)
4. Scale the data's features
5. feature selection
6. Apply the training data to the model

* We cannot chain the ColumnTransformer as that will reqiure manual tracking of the columns/features because a ColumnTransformers shift the features' indexes, change their names which will lead another transformer to produce un-wanted results kind of like a concurrency problem.

* So we do: pipelines(with some transforms) -> preprocessor(ColumnTransformer) -> pipepline(preprocessor -> model) pipelines work in a sequential/dependent way i.e. The product is dependent on many processes on after another, the transformations in a ColumnTransformer work separately/independently of each other and only merged in the final product.

1. First, we define which columns/features we want to work on.

    Classifying the features(type of data):

In [ ]:
numerical_features_list = ['Age', 'Fare']
categorial_features_list = ['Sex', 'Embarked']

2. Secondly, we make the pipelines, because we want to impute(fill) the missing values in the features and only then apply the transformations(Scale or One hot encode)

In [ ]:
numerical_pipeline = Pipeline(
    [
        ('impute_age_fare', SimpleImputer()),
        ('scaler_age_fare', StandardScaler())
    ]
)

category_pipeline = Pipeline(
    [
        ('impute_sex_embarked', SimpleImputer(strategy = 'most_frequent')),
        ('ohe_sex_embarked', OneHotEncoder(drop = 'first'))
    ]
)

3. ColumnTransform is defined with a OrdinalEncoder(on features, 'A', 'B') 
and a OneHotEncoder (On Features, 'C', 'D') and is fit_transform to X_train, OrdinalEncoder and OneHotEncoder are given their respective features and when the appropriate transformation is done. The results and the remaining columns are concatenated in order and returned as a NumPy array in preprocessor.

In [ ]:

preprocessor = ColumnTransformer(
    [
        ('numerical', numerical_pipeline, numerical_features_list),
        ('category', category_pipeline, categorial_features_list)
    ]
)


Initializing the pipeline:

In [ ]:

pipe = Pipeline(
    [
        ('preprocessing', preprocessor),
        ('model', DecisionTreeClassifier())
    ]
)

Alternatively, `make_pipeline`
* same thing as `make_column_transformer` for ColumnTransformer

In [ ]:
pipe2 = make_pipeline(preprocessor, DecisionTreeClassifier())

In [ ]:
pipe.named_steps

In [ ]:
pipe.fit(X = X_train, y = y_train)

In [ ]:
pipe.predict(X_test)


Cross Validation using Pipeline

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe, X_train, y_train, cv = 5, scoring = 'accuracy').mean()

GridSearch using Pipeline

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'model__max_depth':[1, 2, 3, 4, 5, None]
}
grid = GridSearchCV(pipe, params, cv = 5, scoring = 'accuracy')
grid.fit(X_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
grid.best_params_

Exporting the Pipeline

In [ ]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))